In [ ]:
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
import hashlib
from datetime import datetime, timezone
import os
import time
from tqdm import tqdm

In [ ]:
# Retrieve the Pinecone API key from the environment
pinecone_key = os.environ.get("PINECONE_API_KEY")

In [ ]:
# Initialize the OpenAI client with the API key from the environment
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [ ]:
# Define constants for the Pinecone index, namespace, and engine
INDEX_NAME = "semantic-search-rag"  # The name of the Pinecone index
NAMESPACE = "default"  # The namespace to use within the index
ENGINE = "text-embedding-3-small"  # The embedding model to use (vector size 1,536)

In [ ]:
# Initialize the Pinecone client with the retrieved API key
pc = Pinecone(api_key=pinecone_key)

In [ ]:
# Function to get embeddings for a list of texts using the OpenAI API
def get_embeddings(texts, engine=ENGINE):
    # Create embeddings for the input texts using the specified engine
    response = client.embeddings.create(input=texts, model=engine)

    return [d.embedding for d in list(response.data)]

In [ ]:
# Function to get embedding for a single text using the OpenAI API
def get_embedding(text, engine=ENGINE):
    # Use the get_embeddings function to get the embedding for a single text
    return get_embeddings([text], engine)[0]

In [ ]:
# Test the functions by getting the length of a single embedding and a list of embeddings
print(len(get_embedding("hi")), len(get_embeddings(["hi", "hello"])))  # (1536, 2)

In [ ]:
if INDEX_NAME not in pc.list_indexes().names():  # need to create the index
    print(f"Creating index {INDEX_NAME}")
    pc.create_index(
        name=INDEX_NAME,  # The name of the index
        dimension=1536,  # The dimensionality of the vectors for our OpenAI embedder
        metric="cosine",  # The similarity metric to use when searching the index
        spec=ServerlessSpec(cloud="aws", region="us-west-2"),
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

In [ ]:
# Store the index as a variable
index = pc.Index(name=INDEX_NAME)
print(index)

In [ ]:
print(index.describe_index_stats())

In [ ]:
def my_hash(s):
    # Return the MD5 hash of the input string as a hexadecimal string
    return hashlib.md5(s.encode()).hexdigest()

In [ ]:
print(my_hash("I love to hash it"))  # ae76cc4dfd345ecaeea9b8ba0d5c3437

In [ ]:
def prepare_for_pinecone(texts, engine=ENGINE, urls=None):
    # Get the current UTC date and time
    now = datetime.now(
        timezone.utc
    ).isoformat()  # Pinecone metadata must be JSON-serializable

    # Generate vector embeddings for each string in the input list, using the specified engine
    embeddings = get_embeddings(texts, engine=engine)

    # Create tuples of (hash, embedding, metadata) for each input string and its corresponding vector embedding
    # The my_hash() function is used to generate a unique hash for each string, and the datetime is used to generate the current UTC date and time
    responses = [
        (
            my_hash(
                text
            ),  # A unique ID for each string, generated using the my_hash() function
            embedding,  # The vector embedding of the string
            dict(
                text=text, date_uploaded=now
            ),  # A dictionary of metadata, including the original text and the current UTC date and time
        )
        for text, embedding in zip(
            texts, embeddings
        )  # Iterate over each input string and its corresponding vector embedding
    ]

    if urls and len(urls) == len(texts):
        for response, url in zip(responses, urls):
            response[-1]["url"] = url

    return responses

In [ ]:
texts = ["hi"]

In [ ]:
_id, embedding, metadata = prepare_for_pinecone(texts)[0]

In [ ]:
print("ID: ", _id, "\nLEN: ", len(embedding), "\nMETA:", metadata)

In [ ]:
urls = ["fake.url"]
_id, embedding, metadata = prepare_for_pinecone(texts, urls=urls)[0]

In [ ]:
print("ID: ", _id, "\nLEN: ", len(embedding), "\nMETA:", metadata)

In [ ]:
def upload_texts_to_pinecone(
    texts, namespace=NAMESPACE, batch_size=None, show_progress_bar=False, urls=None
):
    # Call the prepare_for_pinecone function to prepare the input texts for indexing
    total_upserted = 0
    if not batch_size:
        batch_size = len(texts)

    _range = range(0, len(texts), batch_size)
    for i in tqdm(_range) if show_progress_bar else _range:
        text_batch = texts[i : i + batch_size]
        if urls:
            url_batch = urls[i : i + batch_size]
            prepared_texts = prepare_for_pinecone(text_batch, urls=url_batch)
        else:
            prepared_texts = prepare_for_pinecone(text_batch)

        # Use the upsert() method of the index object to upload the prepared texts to Pinecone
        total_upserted += index.upsert(vectors=prepared_texts, namespace=namespace)[
            "upserted_count"
        ]

    return total_upserted

In [ ]:
# Call the upload_texts_to_pinecone() function with the input texts
upload_texts_to_pinecone(texts)

In [ ]:
print(index.describe_index_stats())

In [ ]:
print(texts)

In [ ]:
def query_from_pinecone(query, top_k=3, include_metadata=True):
    # get embedding from THE SAME embedder as the documents
    query_embedding = get_embedding(query, engine=ENGINE)

    return index.query(
        vector=query_embedding,
        top_k=top_k,
        namespace=NAMESPACE,
        include_metadata=include_metadata,  # gets the metadata (dates, text, etc)
    ).get("matches")

In [ ]:
# test that the index is empty
print(query_from_pinecone("hello"))

In [ ]:
def delete_texts_from_pinecone(texts, namespace=NAMESPACE):
    # Compute the hash (id) for each text
    hashes = [hashlib.md5(text.encode()).hexdigest() for text in texts]

    # The ids parameter is used to specify the list of IDs (hashes) to delete
    return index.delete(ids=hashes, namespace=namespace)

In [ ]:
# delete our test
delete_texts_from_pinecone(texts)

In [ ]:
print(index.describe_index_stats())

In [ ]:
base_url = "https://faq.ssa.gov"
medicare_faqs = base_url + "/en-US/topic/?id=CAT-01092"
print(medicare_faqs)

In [ ]:
from bs4 import BeautifulSoup
import requests

In [ ]:
# get all links from medicare_faqs
urls = []
r = requests.get(medicare_faqs)
soup = BeautifulSoup(r.content, "html.parser")
for link in soup.find_all("a"):
    if "href" in link.attrs:
        if link["href"].startswith("/") and "article" in link["href"]:
            urls.append(base_url + link["href"])

In [ ]:
print(urls)

In [ ]:
texts = []
for url in tqdm(urls):
    r = requests.get(url)
    soup = BeautifulSoup(r.content, "html.parser")
    body = soup.find("body").get_text()
    texts.append(body)

In [ ]:
# CLEAN YOUR DATA HERE :)

In [ ]:
print(texts[0] if texts else "No texts scraped")

In [ ]:
BATCH_SIZE = 4
print(
    upload_texts_to_pinecone(
        texts, batch_size=BATCH_SIZE, urls=urls, show_progress_bar=True
    )
)

In [ ]:
print(index.describe_index_stats())

In [ ]:
results = query_from_pinecone("I lost my Medicare card")
for result in results:
    print(result["metadata"]["url"], result["score"], result["metadata"]["text"][:50])